# Mesh 8 · A mesh is a truss: forces in every bar

**Application.** Companion to [Week 05 §12](../README.md#12-mesh-in-architecture-four-applications)
and `mesh_applications/structural_analysis.py`. Uses notebooks 03 and 04.

![The hanging net in tension, the inverted vault in compression, and a pyramid of the same rise that fails as a mechanism](../figures/app-structure.svg)

Edges are bars, vertices are pin joints. The script solves a whole vault in
one call; here the method is built from **one bar** upwards, so that the big
solve at the end holds no surprises.

Do [`07_form_finding`](07_form_finding.ipynb) first: this notebook checks the
shape that one finds.

In [ ]:
# Setup: run this cell first. It finds the shared drawing helper (viz.py),
# whether this notebook sits in Lecture/notebooks/ or is your copy in MyWork/.
import sys
import pathlib

HERE = pathlib.Path.cwd()
NB = next(p / "Lecture" / "notebooks" for p in [HERE, *HERE.parents]
          if (p / "Lecture" / "notebooks" / "viz.py").exists())
sys.path.insert(0, str(NB))
from viz import chart, draw_graph, draw_mesh, draw_points, ramp, row, view3d

import numpy as np
from compas.datastructures import Mesh

np.set_printoptions(precision=2, suppress=True)

## Step 1 — one bar is a 3 × 3 matrix
A bar only resists being stretched **along its own axis**. With `e` the unit
vector along the bar, its stiffness is `EA / L * outer(e, e)`: push along
`e` and it pushes back; push sideways and it does nothing.

In [ ]:
EA = 1.0e4                       # axial stiffness, the same for every bar


def bar_stiffness(a, b):
    d = np.array(b, float) - np.array(a, float)
    L = np.linalg.norm(d)
    e = d / L
    return EA / L * np.outer(e, e), e, L


k, e, L = bar_stiffness([0, 0, 0], [2, 0, 0])
print("a bar along x, 2 m long:\n", k)
print("pushed along x: ", k @ [0.001, 0, 0])
print("pushed sideways:", k @ [0, 0.001, 0], " <- no resistance at all")

## Step 2 — a tripod: three bars, one free joint
Three feet on the ground (pinned), one apex (free), a load pulling the apex
down. The apex is held by all three bars, so its stiffness is the **sum** of
the three bar matrices. Then `K u = f` is three equations.

In [ ]:
tripod = Mesh.from_vertices_and_faces(
    [[2, 0, 0], [-1, 3 ** 0.5, 0], [-1, -(3 ** 0.5), 0], [0, 0, 3]],
    [[0, 1, 3], [1, 2, 3], [2, 0, 3]])
apex, feet = 3, [0, 1, 2]

K = np.zeros((3, 3))
for foot in feet:
    k, e, L = bar_stiffness(tripod.vertex_coordinates(foot), tripod.vertex_coordinates(apex))
    K += k
f = np.array([0.0, 0.0, -9.0])                 # 9 kN down
u = np.linalg.solve(K, f)
print("K of the apex:\n", K)
print("apex moves by", u * 1000, "mm")
draw_mesh(tripod, "the tripod", view="iso")

### From movement to force
A bar's force is its stiffness times its **stretch**: how much the apex
moved along the bar's own axis. Negative means shorter: compression.

In [ ]:
total_up = 0.0
for foot in feet:
    k, e, L = bar_stiffness(tripod.vertex_coordinates(foot), tripod.vertex_coordinates(apex))
    force = EA / L * (e @ u)
    total_up += -force * e[2]
    print(f"bar {foot}-{apex}: force {force:6.2f} kN")
print(f"vertical components add up to {total_up:.2f} kN: the load, carried")

## Step 3 — the same loop for any mesh
Nothing new: every vertex gets three rows of `K`, every edge adds its 3 × 3
block in four places (both ends, and the coupling between them). Supports
are rows we do not solve for.

One addition matters more than the rest: **the check before the solve.** If
some joint can move without stretching any bar, `K` is singular.
`np.linalg.solve` would still return numbers, enormous and meaningless. A
solver that cannot fail loudly is not a verification tool, so this one
raises.

In [ ]:
def analyse(mesh, supports, load=-1.0):
    """Bar forces {edge: force} and displacements {vertex: [dx, dy, dz]}."""
    keys = list(mesh.vertices())
    index = {key: i for i, key in enumerate(keys)}
    K = np.zeros((3 * len(keys), 3 * len(keys)))
    bars = []
    for a, b in mesh.edges():
        k, e, L = bar_stiffness(mesh.vertex_coordinates(a), mesh.vertex_coordinates(b))
        i, j = index[a], index[b]
        for p, q, sign in ((i, i, 1), (j, j, 1), (i, j, -1), (j, i, -1)):
            K[3 * p:3 * p + 3, 3 * q:3 * q + 3] += sign * k
        bars.append(((a, b), i, j, e, L))

    f = np.zeros(3 * len(keys))
    for key in keys:
        f[3 * index[key] + 2] = load * mesh.vertex_area(key)      # load per m2 of surface

    free = [3 * index[key] + c for key in keys if key not in set(supports) for c in range(3)]
    Kff = K[np.ix_(free, free)]
    if np.linalg.cond(Kff) > 1e12:
        raise ValueError("mechanism: some joint can move without stretching any bar")
    u = np.zeros(3 * len(keys))
    u[free] = np.linalg.solve(Kff, f[free])

    forces = {edge: float(EA / L * (e @ (u[3 * j:3 * j + 3] - u[3 * i:3 * i + 3])))
              for edge, i, j, e, L in bars}
    moves = {key: u[3 * index[key]:3 * index[key] + 3] for key in keys}
    return forces, moves


forces, moves = analyse(tripod, feet, load=-9.0 / tripod.vertex_area(apex))
print("the tripod again, through the general function:")
print({edge: round(force, 2) for edge, force in forces.items() if apex in edge})

## Step 4 — three roofs over the same square
A 10 m square, supported all round. Three shapes with the same rise:

- a **hanging net**, found by the force density method (notebook
  `07_form_finding` unpacks these few lines)
- the net **turned upside down**: a vault
- a **pyramid** with flat sides

In [ ]:
def form_find(mesh, fixed, q=2.0, load=-1.0):
    """Force density method, in short. Returns a hanging copy of the mesh."""
    keys = list(mesh.vertices())
    index = {key: i for i, key in enumerate(keys)}
    C = np.zeros((mesh.number_of_edges(), len(keys)))
    for r, (a, b) in enumerate(mesh.edges()):
        C[r, index[a]], C[r, index[b]] = 1, -1
    X = np.array([mesh.vertex_coordinates(key) for key in keys])
    fr = [index[key] for key in keys if key not in fixed]
    fx = [index[key] for key in fixed]
    P = np.zeros_like(X)
    P[:, 2] = load
    Q = q * np.eye(mesh.number_of_edges())
    X[fr] = np.linalg.solve(C[:, fr].T @ Q @ C[:, fr], P[fr] - C[:, fr].T @ Q @ C[:, fx] @ X[fx])
    result = mesh.copy()
    for key, xyz in zip(keys, X):
        result.vertex_attributes(key, "xyz", [float(c) for c in xyz])
    return result


flat = Mesh.from_meshgrid(dx=10, nx=8)
rim = [v for v in flat.vertices() if flat.is_vertex_on_boundary(v)]

net = form_find(flat, rim)
rise = -min(net.vertices_attribute("z"))

vault = net.copy()
pyramid = flat.copy()
for v in flat.vertices():
    x, y, z = net.vertex_coordinates(v)
    vault.vertex_attribute(v, "z", -z)
    pyramid.vertex_attribute(v, "z", rise * (1 - max(abs(x - 5), abs(y - 5)) / 5))

shapes = {"hanging net": net, "vault": vault, "pyramid": pyramid}
print(f"rise {rise:.2f} m over a 10 m span")
row(*[draw_mesh(m, name, view="iso", vertex_labels=False, size=(330, 280)) for name, m in shapes.items()])

### Quads first: the solver refuses
These meshes are quads. Four pin-jointed bars can fold into a rhombus
without any bar changing length, so the solver must refuse.

In [ ]:
try:
    analyse(vault, rim)
except ValueError as error:
    print("vault with quad faces ->", error)

### Triangulate, then solve all three
A diagonal in every quad removes the folding. Then read one number per
roof: of all the force in the bars, how much is compression?

In [ ]:
results = {}
for name, shape in shapes.items():
    tri = shape.copy()
    tri.quads_to_triangles()
    try:
        forces, moves = analyse(tri, rim)
    except ValueError as error:
        print(f"{name:12s} {error}")
        continue
    values = np.array(list(forces.values()))
    share = -values[values < 0].sum() / np.abs(values).sum()
    sag = max(np.linalg.norm(m) for m in moves.values())
    results[name] = (tri, forces, moves)
    print(f"{name:12s} compression share {share:6.1%}   largest bar force {np.abs(values).max():5.2f}   "
          f"largest movement {sag * 1000:5.2f} mm")

Three results, three lessons:

- Hooke, 1675: *as hangs the flexible line, so but inverted will stand the
  rigid arch.* The net is nearly all tension, and the same shape upside down
  is nearly all compression: a shape stone can be built in.
- *Nearly*, not exactly: the diagonals added for rigidity take a share of
  the load. Measure, do not assume.
- The pyramid has the same rise and the same bars, and it does not stand at
  all. Its flat sides can move in and out freely, so it is a mechanism.
  **Shape, not material, decides.**

## Step 5 — put the forces back on the mesh
Store each bar's force as an edge attribute and the result travels with the
geometry. Below, the bars in **tension** are orange. The bars along the rim
join two supports, so they carry nothing in either roof.

In [ ]:
figures = []
for name, (tri, forces, moves) in results.items():
    for edge, force in forces.items():
        tri.edge_attribute(edge, "force", force)
    for v, move in moves.items():
        tri.vertex_attribute(v, "sag", float(np.linalg.norm(move)))
    pulled = list(tri.edges_where_predicate(lambda edge, attr: attr["force"] > 1e-9))
    print(f"{name:12s} {len(pulled):3d} of {tri.number_of_edges()} bars in tension")
    figures.append(draw_mesh(tri, f"{name}: tension in orange", vertex_labels=False,
                             highlight_edges=pulled, size=(440, 400)))
row(*figures)

In [ ]:
tri, forces, moves = results["vault"]
worst = max(tri.vertices(), key=lambda v: tri.vertex_attribute(v, "sag"))
print(f"the vault moves most at vertex {worst}: {tri.vertex_attribute(worst, 'sag') * 1000:.2f} mm")
view3d((tri, {"show_edges": True}))

## Your turn
1. Support the vault at its four corners only (`vertex_degree(v) == 2`).
   Does it still solve? What happens to the largest bar force?
2. Halve `EA`. Which outputs change and which do not? Why is that what you
   would expect from a structure whose forces follow from equilibrium?
3. Give the pyramid a slight bulge (add `0.3 * sin` of something to its
   `z`). At what bulge does the solver stop refusing?
4. Compare `analyse` with `analyse_truss` in
   `mesh_applications/structural_analysis.py`. What does the
   script do that this notebook leaves out?